# Hardware-aware grid-search database

Run the setup cell, then each table cell to inspect the current results.

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd
from IPython.display import display

data_dir = Path.cwd() if Path.cwd().name == 'data' else Path.cwd() / 'data'
database_path = data_dir / 'hardware_aware_fake_torino_full_2026-10-04.db'

if not database_path.exists():
    raise FileNotFoundError(f'Database not found: {database_path}')

connection = sqlite3.connect(database_path)
print(f'Opened: {database_path}')

## Run statistics

In [ ]:
# Treat each circuit/optimizer pair as one job and use its latest attempt.
run_stats_df = pd.read_sql_query("""
    WITH latest_runs AS (
        SELECT r.*, ROW_NUMBER() OVER (
            PARTITION BY circuit_id, optimizer_id
            ORDER BY created_at DESC, id DESC
        ) AS row_number
        FROM optimization_runs AS r
    )
    SELECT
        COUNT(*) AS jobs,
        (SELECT COUNT(*) FROM optimization_runs) AS total_attempts,
        SUM(CASE WHEN success = 1 THEN 1 ELSE 0 END) AS successful,
        SUM(CASE WHEN success = 0 THEN 1 ELSE 0 END) AS failed,
        SUM(CASE WHEN json_extract(metadata_json, '$.equivalence_verified') = 1 THEN 1 ELSE 0 END) AS verified,
        SUM(CASE WHEN artifact_path IS NOT NULL THEN 1 ELSE 0 END) AS with_artifacts,
        ROUND(100.0 * AVG(success), 2) AS success_rate_percent,
        ROUND(100.0 * AVG(CASE WHEN json_extract(metadata_json, '$.equivalence_verified') = 1 THEN 1.0 ELSE 0.0 END), 2) AS verification_rate_percent
    FROM latest_runs
    WHERE row_number = 1
""", connection)
display(run_stats_df.T.rename(columns={0: 'value'}))

status_by_optimizer_df = pd.read_sql_query("""
    WITH latest_runs AS (
        SELECT r.*, ROW_NUMBER() OVER (
            PARTITION BY circuit_id, optimizer_id
            ORDER BY created_at DESC, id DESC
        ) AS row_number
        FROM optimization_runs AS r
    ), attempts AS (
        SELECT optimizer_id, COUNT(*) AS total_attempts
        FROM optimization_runs
        GROUP BY optimizer_id
    )
    SELECT
        o.name AS optimizer,
        COUNT(r.id) AS jobs,
        MAX(a.total_attempts) AS total_attempts,
        SUM(CASE WHEN r.success = 1 THEN 1 ELSE 0 END) AS successful,
        SUM(CASE WHEN r.success = 0 THEN 1 ELSE 0 END) AS failed,
        SUM(CASE WHEN json_extract(r.metadata_json, '$.equivalence_verified') = 1 THEN 1 ELSE 0 END) AS verified,
        ROUND(100.0 * AVG(r.success), 2) AS success_rate_percent
    FROM latest_runs AS r
    JOIN optimizers AS o ON o.id = r.optimizer_id
    JOIN attempts AS a ON a.optimizer_id = r.optimizer_id
    WHERE r.row_number = 1
    GROUP BY o.id, o.name
    ORDER BY o.name
""", connection)
display(status_by_optimizer_df)

## Circuits

In [ ]:
circuits_df = pd.read_sql_query('SELECT * FROM circuits ORDER BY name', connection)
display(circuits_df)

## Optimizers

In [ ]:
optimizers_df = pd.read_sql_query('SELECT * FROM optimizers ORDER BY name', connection)
display(optimizers_df)

## Optimization runs

In [ ]:
optimization_runs_df = pd.read_sql_query('SELECT * FROM optimization_runs ORDER BY id', connection)
display(optimization_runs_df)

## Trajectories

In [ ]:
trajectories_df = pd.read_sql_query('SELECT * FROM trajectories ORDER BY id', connection)
display(trajectories_df)

## Trajectory steps

In [ ]:
trajectory_steps_df = pd.read_sql_query('SELECT * FROM trajectory_steps ORDER BY id', connection)
display(trajectory_steps_df)